# Detecção de objetos com YOLOv8

Este notebook demonstra um fluxo simples de detecção de objetos com YOLOv8 no Google Colab.

A proposta é usar um modelo pré-treinado, executar inferência em uma imagem de exemplo, salvar o resultado com bounding boxes e documentar como seria o caminho para transfer learning com uma base customizada.

Importante: este notebook não treina um modelo do zero. A parte prática usa pesos pré-treinados do YOLOv8. A seção de treinamento customizado mostra o caminho para quando houver uma base anotada real.

## 1. Instalação do Ultralytics

A biblioteca Ultralytics facilita o uso do YOLOv8 no Colab. A instalação abaixo baixa o pacote necessário para carregar modelos, executar inferência e treinar com datasets no formato YOLO.

In [ ]:
!pip install ultralytics -q

## 2. Importações

Vamos importar as bibliotecas usadas no notebook. O `YOLO` carrega o modelo, o `Path` ajuda a organizar os caminhos e o `matplotlib` mostra o resultado na tela.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

## 3. Organização das pastas

O resultado final será salvo em `images/output`, seguindo a mesma estrutura do projeto no GitHub.

In [ ]:
PROJECT_DIR = Path('/content/yolo-object-detection-dio')
INPUT_DIR = PROJECT_DIR / 'images' / 'input'
OUTPUT_DIR = PROJECT_DIR / 'images' / 'output'

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Pasta de entrada: {INPUT_DIR}')
print(f'Pasta de saída: {OUTPUT_DIR}')

## 4. Carregamento do modelo YOLO pré-treinado

Vamos usar o `yolov8n.pt`, uma versão leve do YOLOv8. Ela é adequada para demonstrações no Colab porque baixa rápido e executa bem em CPU ou GPU.

Esse modelo foi pré-treinado no COCO Dataset, então já reconhece classes como `person`, `car`, `dog` e `bicycle`.

In [ ]:
model = YOLO('yolov8n.pt')
print('Modelo YOLOv8 carregado com sucesso.')

## 5. Transfer learning

Transfer learning é o uso de um modelo já treinado como ponto de partida para uma nova tarefa.

Em vez de treinar uma rede do zero, aproveitamos os pesos aprendidos em uma base grande, como o COCO, e ajustamos o modelo com uma base menor e específica.

Exemplo: se o objetivo fosse detectar apenas `dog` e `bicycle` em imagens próprias, poderíamos começar com `yolov8n.pt` e treinar por algumas épocas usando uma base anotada com essas duas classes.

## 6. Dataset anotado

Para treinar um detector de objetos, não basta ter as imagens. Também é necessário informar onde cada objeto aparece.

No formato YOLO, cada imagem possui um arquivo `.txt` correspondente com as bounding boxes e a classe de cada objeto.

Uma estrutura comum é:

```text
dataset/
├── images/
│   ├── train/
│   └── val/
├── labels/
│   ├── train/
│   └── val/
└── data.yaml
```

Ferramentas como Labelme podem ser usadas para criar as anotações. Depois, as anotações precisam estar no formato esperado pelo YOLO.

## 7. Classes detectadas

O modelo pré-treinado no COCO reconhece várias classes. Neste projeto, as classes de referência incluem pelo menos duas das seguintes:

- `person`
- `car`
- `dog`
- `bicycle`

A imagem de exemplo usada abaixo contém objetos que o modelo já consegue reconhecer.

## 8. Baixar imagem de exemplo

A imagem abaixo é baixada apenas para demonstrar a inferência. Em um projeto próprio, você pode substituir esse arquivo por uma imagem da sua base.

In [ ]:
example_image_path = INPUT_DIR / 'example.jpg'

!wget -q -O {example_image_path} https://ultralytics.com/images/bus.jpg

print(f'Imagem de exemplo salva em: {example_image_path}')

## 9. Execução de inferência

Agora o modelo analisa a imagem e retorna as detecções encontradas. O parâmetro `conf=0.25` define a confiança mínima para exibir uma detecção.

In [ ]:
results = model.predict(
    source=str(example_image_path),
    conf=0.25,
    save=False,
    verbose=False,
)

detections = results[0]
print(f'Objetos detectados: {len(detections.boxes)}')

## 10. Salvamento do resultado com bounding boxes

A função `plot()` gera uma cópia da imagem com as caixas e nomes das classes desenhados. Depois salvamos o arquivo em `images/output`.

In [ ]:
annotated_image = detections.plot()
output_path = OUTPUT_DIR / 'yolo_result.jpg'

cv2.imwrite(str(output_path), annotated_image)

print(f'Resultado salvo em: {output_path}')

## 11. Visualização do resultado

O OpenCV lê imagens em BGR, enquanto o Matplotlib mostra em RGB. Por isso fazemos a conversão antes de exibir.

In [ ]:
image_bgr = cv2.imread(str(output_path))
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(image_rgb)
plt.axis('off')
plt.title('Resultado da detecção com YOLOv8')
plt.show()

## 12. Como seria o treinamento customizado com duas classes

Para treinar um modelo com classes próprias, é necessário preparar um dataset real com imagens e labels.

Exemplo de classes:

- `dog`
- `bicycle`

O arquivo `data.yaml` poderia ficar assim:

```yaml
path: /content/yolo-object-detection-dio/dataset
train: images/train
val: images/val

names:
  0: dog
  1: bicycle
```

Com a base organizada, o YOLOv8 pode ajustar os pesos do modelo pré-treinado para reconhecer melhor essas classes no contexto do dataset customizado.

## 13. Comandos comentados para treinamento customizado

Os comandos abaixo ficam comentados porque exigem um dataset real no formato YOLO. Eles mostram o caminho correto para aplicar transfer learning quando a base estiver pronta.

In [ ]:
# Exemplo de treinamento com dataset customizado no formato YOLO.
# Execute somente depois de criar dataset/data.yaml e organizar imagens e labels.

# !yolo detect train model=yolov8n.pt data=/content/yolo-object-detection-dio/dataset/data.yaml epochs=30 imgsz=640

# Depois do treinamento, o melhor peso costuma ficar em:
# runs/detect/train/weights/best.pt

# Exemplo de inferência com o modelo treinado:
# !yolo detect predict model=runs/detect/train/weights/best.pt source=/content/yolo-object-detection-dio/images/input

## 14. Conclusão

Este notebook executa um fluxo funcional de detecção de objetos com YOLOv8 no Colab.

Foi usado um modelo pré-treinado para gerar uma imagem com bounding boxes e salvar o resultado em `images/output`.

Também foi documentado como preparar uma base anotada e como iniciar um treinamento customizado por transfer learning com pelo menos duas classes. Essa etapa depende de imagens e labels reais, por isso os comandos foram deixados como referência comentada.